# Vesper-IndicEAR quickstart

Transcribe speech in 11 Indian languages and English with Vesper-IndicEAR (part of the Vesper project): a frozen EmbeddingGemma 2 audio encoder with small trained adapters and heads. The output is lowercase text without punctuation, each word in its own script (Latin for English words), plus the detected language.

Runs on a CPU, an NVIDIA GPU or Apple MPS. The model download is 627 MiB (it includes the encoder's audio tower).

In [ ]:
# @title 1. Install
!pip install -q indicear

In [ ]:
# @title 2. Load the model (downloads it once)
from indicear.transcribe import Transcriber

model = Transcriber("rahulsharmavishwakarma/vesper-indicEAR")  # device and precision are picked automatically
print(model.manifest["languages"], model.device)

In [ ]:
# @title 3. An English sample (LibriSpeech, CC BY 4.0)
import io, soundfile as sf
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

parquet = hf_hub_download("hf-internal-testing/librispeech_asr_dummy", "clean/validation-00000-of-00001.parquet",
                          repo_type="dataset", revision="5be91486e11a2d616f4ec5db8d3fd248585ac07a")
row = pq.read_table(parquet).slice(0, 1).to_pylist()[0]
audio, rate = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
sf.write("sample.wav", audio, rate)
print("reference:", row["text"].lower())

In [ ]:
# @title 4. Transcribe a file
result = model.transcribe_file("sample.wav")
print(result["text"])
print("language:", result["language"]["top"])

In [ ]:
# @title 5. Your own audio: upload any file soundfile or ffmpeg can read (wav, flac, mp3, m4a, ...)
# In Colab: from google.colab import files; name = next(iter(files.upload()))
# result = model.transcribe_file(name)
# If you know the language, a hint helps (most of all for Urdu, which is often detected as Hindi):
# from indicear.transcribe import read_audio
# result = model.transcribe(read_audio(name), lang_hint="ur")

In [ ]:
# @title 6. Words, times and code-switch spans
for word in result["words"][:10]:
    print(f'{word["start"]:6.2f} {word["end"]:6.2f}  {word["word"]}  {word.get("lang", "")}')
model.close()